In [45]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_URL  = "https://data.cityofchicago.org/resource/ajtu-isnz.json"
WHERE     = "trip_start_timestamp >= '2026-02-01T00:00:00'"
PAGE_SIZE = 50_000   # Socrata hard cap is 50,000 per request — use the maximum
WORKERS   = 8        # parallel threads

# ── Step 1: get total row count first (one cheap query) ──────────────────────
resp = requests.get(
    BASE_URL,
    params={"$select": "count(*)", "$where": WHERE},
    timeout=30,
)
resp.raise_for_status()
total = int(resp.json()[0]["count"])
print(f"Total rows to fetch: {total:,}")

# ── Step 2: build offset list ─────────────────────────────────────────────────
offsets = list(range(0, total, PAGE_SIZE))
print(f"Pages to fetch: {len(offsets)}  ×  {PAGE_SIZE:,} rows each")

# ── Step 3: fetch all pages in parallel ──────────────────────────────────────
def fetch_page(offset):
    r = requests.get(
        BASE_URL,
        params={
            "$where":  WHERE,
            #"$order":  "trip_start_timestamp ASC",
            "$limit":  PAGE_SIZE,
            "$offset": offset,
        },
        timeout=60,
    )
    r.raise_for_status()
    return r.json()

results = [None] * len(offsets)   # pre-allocated so order is preserved

with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    future_to_idx = {pool.submit(fetch_page, off): i for i, off in enumerate(offsets)}
    completed = 0
    for future in as_completed(future_to_idx):
        idx = future_to_idx[future]
        results[idx] = future.result()
        completed += 1
        rows_so_far = sum(len(r) for r in results if r is not None)
        print(f"  {completed}/{len(offsets)} pages done — {rows_so_far:,} rows fetched")

# ── Step 4: flatten and build DataFrame ──────────────────────────────────────
all_rows = [row for page in results for row in page]
df = pd.DataFrame(all_rows)
print(f"\nDone — {len(df):,} rows  ×  {df.shape[1]} columns")
print(df.dtypes)

Total rows to fetch: 450,184
Pages to fetch: 10  ×  50,000 rows each
  1/10 pages done — 50,000 rows fetched
  2/10 pages done — 100,000 rows fetched
  3/10 pages done — 150,000 rows fetched
  4/10 pages done — 200,000 rows fetched
  5/10 pages done — 200,184 rows fetched
  6/10 pages done — 250,184 rows fetched
  7/10 pages done — 300,184 rows fetched
  8/10 pages done — 350,184 rows fetched
  9/10 pages done — 400,184 rows fetched
  10/10 pages done — 450,184 rows fetched

Done — 450,184 rows  ×  23 columns
trip_id                       object
taxi_id                       object
trip_start_timestamp          object
trip_end_timestamp            object
trip_seconds                  object
trip_miles                    object
pickup_community_area         object
dropoff_community_area        object
fare                          object
tips                          object
tolls                         object
extras                        object
trip_total                    object
paym

In [53]:
df = pd.DataFrame(all_rows)
display(df)

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area,dropoff_community_area,fare,tips,...,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,pickup_centroid_location,dropoff_centroid_latitude,dropoff_centroid_longitude,dropoff_centroid_location,pickup_census_tract,dropoff_census_tract
0,b291af5e326d1047c8241e3c21810dec7d7614f1,342474dd3a24c9d5b7dbc2514032a01ecb396624638be5...,2026-03-01T00:00:00.000,2026-03-01T00:15:00.000,1680,5.6,32,24,16.75,0,...,Cash,Transit Administrative Center Inc,41.878865584,-87.625192142,"{'type': 'Point', 'coordinates': [-87.62519214...",41.901206994,-87.676355989,"{'type': 'Point', 'coordinates': [-87.67635598...",NaN,NaN
1,673f05c48f4ead3be1693051c7a21d7ca0434822,01480513a7f6f4664d6cc4c41ff3043ae6ecbc8cb17404...,2026-03-01T00:00:00.000,2026-03-01T00:00:00.000,669,1.34,8,28,9.52,0,...,Mobile,Blue Ribbon Taxi Association,41.899602111,-87.633308037,"{'type': 'Point', 'coordinates': [-87.63330803...",41.874005383,-87.66351755,"{'type': 'Point', 'coordinates': [-87.66351754...",NaN,NaN
2,65b3ee3f56ca1afd5fff3ddc02b3716032de559c,dff8b2296e1abd3c3470a2f0765b3f385f9a53c8df1cc3...,2026-03-01T00:00:00.000,2026-03-01T00:15:00.000,572,3.07,8,24,13.26,0,...,Mobile,Chicago Independents,41.899602111,-87.633308037,"{'type': 'Point', 'coordinates': [-87.63330803...",41.901206994,-87.676355989,"{'type': 'Point', 'coordinates': [-87.67635598...",NaN,NaN
3,b7c311475e4b2b39a6df75813a370fa908da818b,8c5b237e8b27de4ba2c9d903f8df2ada08258d65eeb856...,2026-03-01T00:00:00.000,2026-03-01T00:00:00.000,251,0.38,8,8,4.75,2,...,Credit Card,Taxicab Insurance Agency Llc,41.899602111,-87.633308037,"{'type': 'Point', 'coordinates': [-87.63330803...",41.899602111,-87.633308037,"{'type': 'Point', 'coordinates': [-87.63330803...",NaN,NaN
4,a4a1ba538589647f965f4465a07c0a5e41fa880b,7a6bcf293f04066d3587fcd6fe2d796bb3fe4ff200f047...,2026-03-01T00:00:00.000,2026-03-01T00:30:00.000,1782,17.93,76,32,44.75,10.05,...,Credit Card,Taxicab Insurance Agency Llc,41.980264315,-87.913624596,"{'type': 'Point', 'coordinates': [-87.91362459...",41.878865584,-87.625192142,"{'type': 'Point', 'coordinates': [-87.62519214...",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450179,3d1534a0f573f8c9e5be50c11b45fe7b270fdf3c,01480513a7f6f4664d6cc4c41ff3043ae6ecbc8cb17404...,2026-02-01T00:00:00.000,2026-02-01T00:15:00.000,858,2.83,8,28,12.44,0,...,Mobile,Blue Ribbon Taxi Association,41.899602111,-87.633308037,"{'type': 'Point', 'coordinates': [-87.63330803...",41.874005383,-87.66351755,"{'type': 'Point', 'coordinates': [-87.66351754...",NaN,NaN
450180,0c286a6e9df6438a54bb66b1650c62c9d6ead129,342474dd3a24c9d5b7dbc2514032a01ecb396624638be5...,2026-02-01T00:00:00.000,2026-02-01T00:15:00.000,1013,0,8,33,13,0,...,Cash,Tac - American United Dispatch,41.899602111,-87.633308037,"{'type': 'Point', 'coordinates': [-87.63330803...",41.857183858,-87.620334624,"{'type': 'Point', 'coordinates': [-87.62033462...",NaN,NaN
450181,dd531229cab8c02d9b7268b51a2eb8bce5e169d6,9dbc9017d66ff7ca48ae4ac833e8fdc0d3f35333516071...,2026-02-01T00:00:00.000,2026-02-01T00:30:00.000,2040,18.6,76,32,46.5,10.2,...,Credit Card,Chicago City Taxi Association,41.980264315,-87.913624596,"{'type': 'Point', 'coordinates': [-87.91362459...",41.878865584,-87.625192142,"{'type': 'Point', 'coordinates': [-87.62519214...",NaN,NaN
450182,46d87c552cf5fe9275ca2f3acd7c5db2edc89682,0fdab9be71f6d88e3d3a2e115afc5a33d2bf74153792c5...,2026-02-01T00:00:00.000,2026-02-01T00:00:00.000,458,1.43,8,8,8.66,0,...,Mobile,City Service,41.899602111,-87.633308037,"{'type': 'Point', 'coordinates': [-87.63330803...",41.899602111,-87.633308037,"{'type': 'Point', 'coordinates': [-87.63330803...",NaN,NaN


In [54]:
df_raw=df

numeric_cols = ['trip_seconds','trip_miles','fare','tips',
                'tolls','extras','trip_total',
                'pickup_centroid_latitude','pickup_centroid_longitude',
                'dropoff_centroid_latitude','dropoff_centroid_longitude']


for col in numeric_cols:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')


#df_raw['trip_start_timestamp'] = pd.to_datetime(df_raw['trip_start_timestamp'], errors='coerce')
#df_raw['trip_end_timestamp']   = pd.to_datetime(df_raw['trip_end_timestamp'],   errors='coerce')
df_raw.columns = [c.lower() for c in df_raw.columns]


print(f'Shape after coercion: {df_raw.shape}')
print(f'Null counts (top 10):\n{df_raw.isnull().sum().nlargest(10)}')


Shape after coercion: (450184, 23)
Null counts (top 10):
dropoff_census_tract          259801
pickup_census_tract           254691
dropoff_community_area         36634
dropoff_centroid_latitude      35464
dropoff_centroid_longitude     35464
dropoff_centroid_location      35464
pickup_community_area          13617
pickup_centroid_latitude       13491
pickup_centroid_longitude      13491
pickup_centroid_location       13491
dtype: int64


In [55]:
display(df_raw)

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area,dropoff_community_area,fare,tips,...,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,pickup_centroid_location,dropoff_centroid_latitude,dropoff_centroid_longitude,dropoff_centroid_location,pickup_census_tract,dropoff_census_tract
0,b291af5e326d1047c8241e3c21810dec7d7614f1,342474dd3a24c9d5b7dbc2514032a01ecb396624638be5...,2026-03-01T00:00:00.000,2026-03-01T00:15:00.000,1680.0,5.60,32,24,16.75,0.00,...,Cash,Transit Administrative Center Inc,41.878866,-87.625192,"{'type': 'Point', 'coordinates': [-87.62519214...",41.901207,-87.676356,"{'type': 'Point', 'coordinates': [-87.67635598...",NaN,NaN
1,673f05c48f4ead3be1693051c7a21d7ca0434822,01480513a7f6f4664d6cc4c41ff3043ae6ecbc8cb17404...,2026-03-01T00:00:00.000,2026-03-01T00:00:00.000,669.0,1.34,8,28,9.52,0.00,...,Mobile,Blue Ribbon Taxi Association,41.899602,-87.633308,"{'type': 'Point', 'coordinates': [-87.63330803...",41.874005,-87.663518,"{'type': 'Point', 'coordinates': [-87.66351754...",NaN,NaN
2,65b3ee3f56ca1afd5fff3ddc02b3716032de559c,dff8b2296e1abd3c3470a2f0765b3f385f9a53c8df1cc3...,2026-03-01T00:00:00.000,2026-03-01T00:15:00.000,572.0,3.07,8,24,13.26,0.00,...,Mobile,Chicago Independents,41.899602,-87.633308,"{'type': 'Point', 'coordinates': [-87.63330803...",41.901207,-87.676356,"{'type': 'Point', 'coordinates': [-87.67635598...",NaN,NaN
3,b7c311475e4b2b39a6df75813a370fa908da818b,8c5b237e8b27de4ba2c9d903f8df2ada08258d65eeb856...,2026-03-01T00:00:00.000,2026-03-01T00:00:00.000,251.0,0.38,8,8,4.75,2.00,...,Credit Card,Taxicab Insurance Agency Llc,41.899602,-87.633308,"{'type': 'Point', 'coordinates': [-87.63330803...",41.899602,-87.633308,"{'type': 'Point', 'coordinates': [-87.63330803...",NaN,NaN
4,a4a1ba538589647f965f4465a07c0a5e41fa880b,7a6bcf293f04066d3587fcd6fe2d796bb3fe4ff200f047...,2026-03-01T00:00:00.000,2026-03-01T00:30:00.000,1782.0,17.93,76,32,44.75,10.05,...,Credit Card,Taxicab Insurance Agency Llc,41.980264,-87.913625,"{'type': 'Point', 'coordinates': [-87.91362459...",41.878866,-87.625192,"{'type': 'Point', 'coordinates': [-87.62519214...",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450179,3d1534a0f573f8c9e5be50c11b45fe7b270fdf3c,01480513a7f6f4664d6cc4c41ff3043ae6ecbc8cb17404...,2026-02-01T00:00:00.000,2026-02-01T00:15:00.000,858.0,2.83,8,28,12.44,0.00,...,Mobile,Blue Ribbon Taxi Association,41.899602,-87.633308,"{'type': 'Point', 'coordinates': [-87.63330803...",41.874005,-87.663518,"{'type': 'Point', 'coordinates': [-87.66351754...",NaN,NaN
450180,0c286a6e9df6438a54bb66b1650c62c9d6ead129,342474dd3a24c9d5b7dbc2514032a01ecb396624638be5...,2026-02-01T00:00:00.000,2026-02-01T00:15:00.000,1013.0,0.00,8,33,13.00,0.00,...,Cash,Tac - American United Dispatch,41.899602,-87.633308,"{'type': 'Point', 'coordinates': [-87.63330803...",41.857184,-87.620335,"{'type': 'Point', 'coordinates': [-87.62033462...",NaN,NaN
450181,dd531229cab8c02d9b7268b51a2eb8bce5e169d6,9dbc9017d66ff7ca48ae4ac833e8fdc0d3f35333516071...,2026-02-01T00:00:00.000,2026-02-01T00:30:00.000,2040.0,18.60,76,32,46.50,10.20,...,Credit Card,Chicago City Taxi Association,41.980264,-87.913625,"{'type': 'Point', 'coordinates': [-87.91362459...",41.878866,-87.625192,"{'type': 'Point', 'coordinates': [-87.62519214...",NaN,NaN
450182,46d87c552cf5fe9275ca2f3acd7c5db2edc89682,0fdab9be71f6d88e3d3a2e115afc5a33d2bf74153792c5...,2026-02-01T00:00:00.000,2026-02-01T00:00:00.000,458.0,1.43,8,8,8.66,0.00,...,Mobile,City Service,41.899602,-87.633308,"{'type': 'Point', 'coordinates': [-87.63330803...",41.899602,-87.633308,"{'type': 'Point', 'coordinates': [-87.63330803...",NaN,NaN


In [ ]:
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

conn = snowflake.connector.connect(
    user="[EMAIL_ADDRESS]",
    password="[PASSWORD]",
    account="[your_account]",
    warehouse="TAXI_WH",
    database="TAXI_DB",
    schema="RAW"
    )

cur  = conn.cursor()


cur.execute('''
    CREATE OR REPLACE TABLE TAXI_DB.RAW.CHICAGO_TRIPS (
        TRIP_ID                    VARCHAR,
        TRIP_START_TIMESTAMP       TIMESTAMP_NTZ,
        TRIP_END_TIMESTAMP         TIMESTAMP_NTZ,
        TRIP_SECONDS               FLOAT,
        TRIP_MILES                 FLOAT,
        PICKUP_COMMUNITY_AREA      VARCHAR,
        DROPOFF_COMMUNITY_AREA     VARCHAR,
        FARE                       FLOAT,
        TIPS                       FLOAT,
        EXTRAS                     FLOAT,
        TRIP_TOTAL                 FLOAT,
        PAYMENT_TYPE               VARCHAR,
        PICKUP_CENTROID_LATITUDE   FLOAT,
        PICKUP_CENTROID_LONGITUDE  FLOAT,
        DROPOFF_CENTROID_LATITUDE  FLOAT,
        DROPOFF_CENTROID_LONGITUDE FLOAT
    )
''')


# Select and rename columns to match DDL
keep = ['trip_id','trip_start_timestamp','trip_end_timestamp','trip_seconds',
        'trip_miles','pickup_community_area','dropoff_community_area','fare',
        'tips','extras','trip_total','payment_type',
        'pickup_centroid_latitude','pickup_centroid_longitude',
        'dropoff_centroid_latitude','dropoff_centroid_longitude']


df_up = df_raw[[c for c in keep if c in df_raw.columns]].copy()
df_up.columns = df_up.columns.str.upper()


success, nchunks, nrows, _ = write_pandas(conn, df_up, 'CHICAGO_TRIPS',
                                           schema='RAW', database='TAXI_DB')
print(f'Loaded: {nrows:,} rows  |  success={success}')
conn.close()


Loaded: 450,184 rows  |  success=True
